# * Mobile Devices

In [2]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

### Step 1 : Import Data Source

In [5]:
''' Rawdata '''

# RAW_MSISDN_DEVICE_NANOCLUSTER-MKS_ARPU.csv
src_file = 'data/RAW_MSISDN_DEVICE_NANOCLUSTER-MKS_ARPU.csv'
src_df = pd.read_csv(src_file)

print(f'\nsrc_df : {src_df.shape[0]} rows, {src_df.shape[1]} columns')
src_df.tail(3)


src_df : 8045317 rows, 32 columns


,CUSTOMER,MSISDN,ACCOUNT_TYPE,ACTIVE_IND,AOU_DEVICE,ARPU_RANGE,CCAATT,SITENAME,REGION,PROVINCE_EN,...,FLAG_DEVICE_IOS-BELOW-IPHONE11,FLAG_4Yearold-luanch_Device,qk16_cen_latitude,qk16_cen_longitude,NN_ID,CLSTR_GRP,TRUEDTAC_MKS,HH,POP,ARPU
8045314,D,66633299855,PREPAID,Y,>18Months,0-99,700701.0,RCB1001,Central-West,Ratchaburi,...,N,N,13.696693,99.846497,700713-001,1,39.96,2105,5979,34.9985
8045315,D,66804381244,PREPAID,Y,>18Months,100-199,570906.0,CRI0517,North,Chiang Rai,...,N,N,20.434734,99.879456,570906-001,6,62.01,15735,34560,185.4206
8045316,D,66629693836,PREPAID,Y,>18Months,300-399,900311.0,SKA0115,South,Songkhla,...,N,Y,6.880073,100.719910,900311-001,4,42.24,1549,5836,329.3637


In [11]:
sample_nano_df = src_df.query(" NN_ID == '210109-001' ").reset_index(drop=True)
sample_nano_df

,CUSTOMER,MSISDN,ACCOUNT_TYPE,ACTIVE_IND,AOU_DEVICE,ARPU_RANGE,CCAATT,SITENAME,REGION,PROVINCE_EN,...,FLAG_DEVICE_IOS-BELOW-IPHONE11,FLAG_4Yearold-luanch_Device,qk16_cen_latitude,qk16_cen_longitude,NN_ID,CLSTR_GRP,TRUEDTAC_MKS,HH,POP,ARPU
0,T,66958191295,POSTPAID,Y,>18Months,0-99,210109.0,RYG0043,East,Rayong,...,N,Y,12.691253,101.214294,210109-001,1,52.71,45640,46965,0.0000
1,T,66638265624,POSTPAID,Y,>18Months,300-399,210109.0,RYG0045,East,Rayong,...,N,Y,12.680535,101.258240,210109-001,1,52.71,45640,46965,300.0000
2,T,66825798524,POSTPAID,Y,>18Months,100-199,210109.0,RYG0085,East,Rayong,...,N,N,12.685894,101.252747,210109-001,1,52.71,45640,46965,100.0000
3,T,66863930698,POSTPAID,Y,>18Months,100-199,210109.0,RYG0201,East,Rayong,...,N,Y,12.685894,101.241760,210109-001,1,52.71,45640,46965,190.0000
4,T,66954954319,POSTPAID,Y,>18Months,0-99,210109.0,RYG0203,East,Rayong,...,N,Y,12.707330,101.175842,210109-001,1,52.71,45640,46965,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11060,D,66993769778,PREPAID,Y,>18Months,600-699,210109.0,RYG0005,East,Rayong,...,Y,N,12.696612,101.186829,210109-001,1,52.71,45640,46965,626.0946
11061,D,66627577439,PREPAID,Y,>18Months,200-299,210109.0,RYG0201,East,Rayong,...,N,N,12.685894,101.241760,210109-001,1,52.71,45640,46965,231.7757
11062,D,66635342328,PREPAID,Y,>18Months,0-99,210109.0,RYG6014,East,Rayong,...,N,N,12.685894,101.241760,210109-001,1,52.71,45640,46965,51.9178
11063,D,66805206526,PREPAID,Y,>18Months,200-299,210109.0,RYG0069,East,Rayong,...,Y,N,12.675176,101.219788,210109-001,1,52.71,45640,46965,289.7288


In [6]:
src_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8045317 entries, 0 to 8045316
Data columns (total 32 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   CUSTOMER                        object 
 1   MSISDN                          int64  
 2   ACCOUNT_TYPE                    object 
 3   ACTIVE_IND                      object 
 4   AOU_DEVICE                      object 
 5   ARPU_RANGE                      object 
 6   CCAATT                          float64
 7   SITENAME                        object 
 8   REGION                          object 
 9   PROVINCE_EN                     object 
 10  DISTRICT_EN                     object 
 11  SUB_DISTRICT_EN                 object 
 12  IMEI_TAC                        int64  
 13  mkt_brand_name                  object 
 14  handset_model                   object 
 15  mkt_model_name                  object 
 16  DVC_OS                          object 
 17  DVC_TYPE                   